In [ ]:
!pip install unsloth mergekit llm-blender
!pip install --no-deps xformers trl peft accelerate bitsandbytes
!pip install torch
!pip install --upgrade trl peft accelerate unsloth
!pip install "transformers<5.5.0"

In [ ]:
from huggingface_hub import login

login(token="ВСТАВЬТЕ_ВАШ_ЛОГИН_СЮДА_ДЛЯ_ДОСТУПА_К_GEMMA3_1B")

In [ ]:
import os
import torch
import pandas as pd
import re
import ast
from tqdm import tqdm
from datasets import Dataset, load_dataset
from unsloth import FastLanguageModel
from trl import SFTTrainer, DPOTrainer 
from transformers import TrainingArguments, TextStreamer
import gc
import json
from collections import Counter

MODEL_NAME = "google/gemma-3-1b-it"
MAX_SEQ_LENGTH = 2048 
DTYPE = torch.bfloat16

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = MODEL_NAME,
    max_seq_length = MAX_SEQ_LENGTH,
    dtype = DTYPE,
    load_in_4bit = False,
)

model = FastLanguageModel.get_peft_model(
    model,
    r = 32,             
    lora_alpha = 64,    
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
)

raw_dataset = load_dataset("HuggingFaceTB/Countdown-Task-GOLD", "verified_Qwen2.5-7B-Instruct", split="train")

In [ ]:
def format_sft(examples):
    texts = []
    for messages in examples["messages"]:
        text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)
        texts.append(text)
    return {"text": texts}

dataset_sft = raw_dataset.select(range(25000)).map(format_sft, batched=True, num_proc=8)

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset_sft,
    dataset_text_field = "text",
    max_seq_length = MAX_SEQ_LENGTH,
    args = TrainingArguments(
        per_device_train_batch_size = 4,
        gradient_accumulation_steps = 16,
        num_train_epochs = 1,
        learning_rate = 2e-4,
        warmup_ratio = 0.05
        lr_scheduler_type = "cosine",
        weight_decay = 0.01,
        bf16 = True,
        optim = "adamw_8bit",
        logging_steps = 50,
        save_strategy = "steps",
        save_steps = 250,
        save_total_limit = 2,
        output_dir = "outputs_sft",
        report_to = "none",
    ),
)

trainer.train()
model.save_pretrained("gemma_sft_final")

In [ ]:
# Идея: генерируем ответы текущей моделью-студентом. Если она ошибается (в математике или тегах),
# мы берем эту ошибку как 'rejected', а идеальный ответ учителя (Qwen) как 'chosen'.
# Это позволяет DPO отучить модель от ее собственных частых ошибок.

FastLanguageModel.for_inference(model)
tokenizer.padding_side = "left"

BATCH_SIZE = 32  
MAX_NEW_TOKENS = 768
OUTPUT_FILE = "train_dpo_gemma_final.jsonl"

def validate_tags(text):
    has_think = bool(re.search(r'<think>.*?</think>', text, re.DOTALL | re.IGNORECASE))
    has_answer = bool(re.search(r'<answer>.*?</answer>', text, re.DOTALL | re.IGNORECASE))
    return has_think and has_answer

def check_math_only(text, target, allowed_nums):
    match = re.search(r'<answer>(.*?)</answer>', text, re.DOTALL | re.IGNORECASE)
    if not match: return False
    
    eq = match.group(1).split('=')[0].strip()
    clean_eq = re.sub(r'[^0-9\+\-\*\/\(\)]', '', eq)
    
    try:
        f_nums = [int(n) for n in re.findall(r'\d+', clean_eq)]
        if Counter(f_nums) != Counter(allowed_nums): return False
        
        return abs(eval(clean_eq, {"__builtins__": None}, {}) - target) < 1e-6
    except: 
        return False

dpo_subset = raw_dataset.select(range(25000, len(raw_dataset)))
dpo_entries = []
hard_negatives = 0
soft_negatives = 0

with open(OUTPUT_FILE, "w", encoding="utf-8") as f_out:
    for i in tqdm(range(0, len(dpo_subset), BATCH_SIZE)):
        batch = dpo_subset.select(range(i, min(i + BATCH_SIZE, len(dpo_subset))))
        current_batch_data = []

        for example in batch:
            messages = example["messages"]
            if not validate_tags(messages[2]["content"]):
                continue
            
            target = example['target']
            nums = ast.literal_eval(example['nums']) if isinstance(example['nums'], str) else example['nums']
            
            prompt = tokenizer.apply_chat_template(messages[:2], tokenize=False, add_generation_prompt=True)
            prompt += "<think>\n"
            
            current_batch_data.append({
                "prompt": prompt,
                "target": target,
                "nums": nums,
                "chosen": messages[2]["content"]
            })

        if not current_batch_data: continue

        prompts_to_gen = [item["prompt"] for item in current_batch_data]
        inputs = tokenizer(prompts_to_gen, return_tensors="pt", padding=True).to("cuda")
        
        with torch.inference_mode():
            outputs = model.generate(
                **inputs, 
                max_new_tokens=MAX_NEW_TOKENS,
                temperature=0.5,
                repetition_penalty=1.2,
                do_sample=True,
                pad_token_id=tokenizer.eos_token_id
            )

        responses = tokenizer.batch_decode(outputs[:, inputs.input_ids.shape[1]:], skip_special_tokens=True)

        for j in range(len(responses)):
            full_resp = "<think>\n" + responses[j].strip()
            data = current_batch_data[j]
            
            has_tags = validate_tags(full_resp)
            is_math_correct = check_math_only(full_resp, data["target"], data["nums"])
            
            if has_tags and is_math_correct:
                continue

            entry = {
                "prompt": data["prompt"],
                "chosen": data["chosen"] + tokenizer.eos_token,
                "rejected": full_resp + tokenizer.eos_token
            }

            if has_tags and not is_math_correct:
                hard_negatives += 1
                f_out.write(json.dumps(entry, ensure_ascii=False) + "\n")
                dpo_entries.append(entry)
            elif not has_tags:
                soft_negatives += 1
                f_out.write(json.dumps(entry, ensure_ascii=False) + "\n")
                dpo_entries.append(entry)
        
        f_out.flush()
        del inputs, outputs
        torch.cuda.empty_cache()
        gc.collect()

In [ ]:
import json
import random

input_file = "train_dpo_gemma_final.jsonl"
output_file = "train_dpo_ULTRA_balanced.jsonl"

hard_examples = []
soft_examples = []

with open(input_file, "r", encoding="utf-8") as f:
    for line in f:
        try:
            item = json.loads(line)
            if "<answer>" in item["rejected"]:
                hard_examples.append(item)
            else:
                soft_examples.append(item)
        except Exception as e:
            continue

final_data = hard_examples + soft_examples

random.shuffle(final_data)

with open(output_file, "w", encoding="utf-8") as f:
    for entry in final_data:
        f.write(json.dumps(entry, ensure_ascii=False) + "\n")

In [ ]:
filtered_data = []
with open("train_dpo_ULTRA_balanced.jsonl", "r") as f:
    for line in f:
        item = json.loads(line)
        len_c = len(item["chosen"])
        len_r = len(item["rejected"])
        
        if len_r < len_c * 2: 
            filtered_data.append(item)

print(f"Осталось нормальных пар: {len(filtered_data)}")

In [ ]:
from trl import DPOConfig, DPOTrainer

dpo_dataset = Dataset.from_list(filtered_data) 

dpo_trainer = DPOTrainer(
    model = model,
    ref_model = None,
    args = DPOConfig(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 8,
        num_train_epochs = 1,
        learning_rate = 5e-6,
        lr_scheduler_type = "cosine",
        bf16 = True,
        optim = "adamw_8bit",
        output_dir = "outputs_dpo_final_hope",
        warmup_ratio = 0.1,
        remove_unused_columns = False,
        loss_type = "ipo", 
    ),
    beta = 1.0,
    train_dataset = dpo_dataset,
    tokenizer = tokenizer,
    max_length = 1024,
    max_prompt_length = 256,
)

dpo_trainer.train()
model.save_pretrained("gemma_3_1b_countdown_ipo_final")
tokenizer.save_pretrained("gemma_3_1b_countdown_ipo_final")

In [ ]:
import os
import torch
import pandas as pd
import re
import ast
from tqdm import tqdm
from datasets import Dataset, load_dataset
from unsloth import FastLanguageModel
from trl import SFTTrainer, DPOTrainer 
from transformers import TrainingArguments, TextStreamer
import gc
import json
from collections import Counter


model_id = "google/gemma-3-1b-it"
MODEL_NAME = "google/gemma-3-1b-it"
MAX_SEQ_LENGTH = 1024
DTYPE = torch.bfloat16

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = MODEL_NAME,
    max_seq_length = MAX_SEQ_LENGTH,
    dtype = DTYPE,
    load_in_4bit = False,
)

model = FastLanguageModel.get_peft_model(
    model,
    r = 32,
    lora_alpha = 64,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
)

In [ ]:
dataset = load_dataset("HuggingFaceTB/Countdown-Task-GOLD", "verified_Qwen3-4B-Instruct-2507", split="train")

tokenizer = get_chat_template(
    tokenizer,
    chat_template = "gemma",
)

def formatting_prompts_func(examples):
    texts = []
    for n, t, msgs in zip(examples["nums"], examples["target"], examples["messages"]):
        instruction = f"Using the numbers {n}, create an equation that equals {t}. Show your work in <think> </think> tags and the final equation in <answer> </answer> tags."
        response = msgs[-1]["content"] if isinstance(msgs, list) else msgs
        
        messages = [
            {"role": "user", "content": instruction},
            {"role": "assistant", "content": response},
        ]
        texts.append(tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=False))
    return { "text" : texts }

dataset = dataset.map(formatting_prompts_func, batched = True)

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    dataset_text_field = "text",
    max_seq_length = 2048,
    dataset_num_proc = 4, 
    args = TrainingArguments(
        per_device_train_batch_size = 32,   
        gradient_accumulation_steps = 4,   
        warmup_ratio = 0.1,                
        num_train_epochs = 1,              
        learning_rate = 1e-4,              
        max_grad_norm = 0.3,               
        fp16 = not is_bfloat16_supported(),
        bf16 = is_bfloat16_supported(),
        logging_steps = 5,
        optim = "adamw_8bit",              
        weight_decay = 0.01,
        lr_scheduler_type = "cosine",      
        seed = 3407,
        output_dir = "gemma_3_1b_sft_fixed",
        save_strategy = "steps",
        save_steps = 500,                  
        dataloader_num_workers = 4,        
        gradient_checkpointing = True,
    ),
)

trainer_stats = trainer.train()

model.save_pretrained("gemma_3_1b_sft_qwen3-4b")
tokenizer.save_pretrained("gemma_3_1b_sft_qwen3-4b")

In [ ]:
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "gemma_3_1b_sft_qwen3-4b",
    max_seq_length = 2048,
    load_in_4bit = False,
)

model.save_pretrained_merged(
    "gemma_dpo_full_weights", 
    tokenizer, 
    save_method = "merged_16bit" 
)

In [ ]:
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "gemma_3_1b_countdown_ipo_final",
    max_seq_length = 2048,
    load_in_4bit = False,
)

model.save_pretrained_merged(
    "gemma_sft_full_weights", 
    tokenizer, 
    save_method = "merged_16bit" 
)

In [ ]:
!mergekit-yaml config.yaml ./gemma_merged --copy-tokenizer --cuda

In [ ]:
from unsloth import FastLanguageModel
import torch

MODEL_PATH = "./gemma_merged"
MAX_SEQ_LENGTH = 2048

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = MODEL_PATH,
    max_seq_length = MAX_SEQ_LENGTH,
    dtype = torch.bfloat16,
    load_in_4bit = True,
    trust_remote_code = True,
)

FastLanguageModel.for_inference(model)
tokenizer.padding_side = "left"

In [ ]:
import torch
import re
import ast
import pandas as pd
import os
import shutil
from tqdm import tqdm
from transformers import AutoTokenizer
from unsloth import FastLanguageModel

MODEL_PATH = "./gemma_merged_ultimate" 
TEST_CSV = "test_public.csv"
OUTPUT_CSV = "submission_final.csv"
N_SAMPLES = 12
BATCH_SIZE = 64
MAX_NEW_TOKENS = 600

src_worker = "./gemma_qwen3_FULL"
files_to_fix = ["preprocessor_config.json", "processor_config.json", "tokenizer.json", "tokenizer_config.json"]
for f in files_to_fix:
    src_f = os.path.join(src_worker, f)
    dst_f = os.path.join(MODEL_PATH, f)
    if os.path.exists(src_f) and not os.path.exists(dst_f):
        shutil.copy2(src_f, dst_f)

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = MODEL_PATH,
    max_seq_length = 2048,
    dtype = torch.bfloat16,
    load_in_4bit = True,
    trust_remote_code = True,
)
FastLanguageModel.for_inference(model)
tokenizer.padding_side = "left"

def strict_verify(eq_str, target, allowed_nums):
    try:
        clean_eq = re.sub(r'[^0-9+\-*/().]', '', str(eq_str))
        if not clean_eq: return False
        
        used_nums = [int(n) for n in re.findall(r'\d+', clean_eq)]
        temp_allowed = list(allowed_nums)
        for n in used_nums:
            if n in temp_allowed:
                temp_allowed.remove(n)
            else:
                return False
        
        return abs(eval(clean_eq) - target) < 1e-4
    except:
        return False

def extract_answer(text):
    match = re.search(r'<answer>(.*?)</answer>', text, re.DOTALL)
    if match:
        ans = match.group(1).strip()
        return ans.split('=')[0].strip()
    return None

def solve_countdown_exact(nums, target):
    def dfs(current_nums, exprs):
        for i, val in enumerate(current_nums):
            if abs(val - target) < 1e-4: return exprs[i]
        if len(current_nums) == 1: return None
        for i in range(len(current_nums)):
            for j in range(len(current_nums)):
                if i == j: continue
                n1, n2, e1, e2 = current_nums[i], current_nums[j], exprs[i], exprs[j]
                next_n_base = [current_nums[k] for k in range(len(current_nums)) if k != i and k != j]
                next_e_base = [exprs[k] for k in range(len(exprs)) if k != i and k != j]
                
                ops = [(n1+n2, f"({e1}+{e2})"), (n1*n2, f"({e1}*{e2})"), (n1-n2, f"({e1}-{e2})")]
                if n2 != 0 and n1 % n2 == 0: ops.append((n1//n2, f"({e1}/{e2})"))
                
                for res_n, res_e in ops:
                    res = dfs(next_n_base + [res_n], next_e_base + [res_e])
                    if res: return res
        return None
    return dfs(nums, [str(n) for n in nums])

df_test = pd.read_csv(TEST_CSV)
if isinstance(df_test['nums'].iloc[0], str):
    df_test['nums'] = df_test['nums'].apply(ast.literal_eval)

final_results = []

for i in tqdm(range(0, len(df_test), BATCH_SIZE)):
    batch = df_test.iloc[i : i + BATCH_SIZE]
    prompts = [
        f"<bos><start_of_turn>user\nSolve Countdown: use {row['nums']} to get {row['target']}.\n"
        f"Format your response with <think> reasoning </think> and <answer> equation </answer>.\n"
        f"<end_of_turn>\n<start_of_turn>model\n<think>\n" 
        for _, row in batch.iterrows()
    ]
    
    inputs = tokenizer(prompts, return_tensors="pt", padding=True).to("cuda")
    
    with torch.inference_mode():
        outputs = model.generate(
            **inputs,
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=True,
            temperature=0.7,
            top_p=0.9,
            repetition_penalty=1.15,
            num_return_sequences=N_SAMPLES,
            use_cache=True,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id
        )
    
    decoded = tokenizer.batch_decode(outputs, skip_special_tokens=True)
    
    for b_idx, (idx, row) in enumerate(zip(batch.index, batch.to_dict('records'))):
        task_id = row['id']
        target = row['target']
        nums = row['nums']
        
        samples = decoded[b_idx * N_SAMPLES : (b_idx + 1) * N_SAMPLES]
        
        found_equation = None
        
        for sample in samples:
            eq = extract_answer(sample)
            if eq and strict_verify(eq, target, nums):
                found_equation = eq
                break
        
        if found_equation is None:
            found_equation = solve_countdown_exact(nums, target)
            if not found_equation:
                found_equation = "0"
        
        final_results.append({"id": task_id, "equation": found_equation})

submission_df = pd.DataFrame(final_results)
submission_df.sort_values("id").to_csv(OUTPUT_CSV, index=False)

/tmp/ipykernel_8974/3867314969.py:9: UserWarning: WARNING: Unsloth should be imported before transformers to ensure all optimizations are applied. Your code may run slower or encounter memory issues without these optimizations.

Please restructure your imports with 'import unsloth' at the top of your file.
  from unsloth import FastLanguageModel


🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!


Unable to import `torchao` Tensor objects. This may affect loading checkpoints serialized with `torchao`


🧠 Загрузка модели и токенизатора...
Unsloth: WARNING `trust_remote_code` is True.
Are you certain you want to do remote code execution?
==((====))==  Unsloth 2025.11.1: Fast Gemma3 patching. Transformers: 4.57.2.
   \\   /|    NVIDIA GeForce RTX 3090. Num GPUs = 1. Max memory: 23.559 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 8.6. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
Unsloth: Gemma3 does not support SDPA - switching to fast eager.
🚀 Старт инференса для 2000 задач...


  6%|▋         | 2/32 [10:45<2:41:02, 322.08s/it]